In [1]:
from google.colab import files
uploaded = files.upload()

Saving raw_customer_sales.csv to raw_customer_sales.csv


```markdown
# Customer Shopping & Sales Analytics

This notebook walks through a full analysis of customer shopping transactions with Pandas, NumPy, Matplotlib, and Seaborn. It covers cleaning the data, then breaking it down by customer, product, city, membership tier, and payment method, followed by sales trends over time, customer segmentation, and a set of charts and correlation checks at the end.

Run the cells in order. You'll need `raw_customer_sales.csv` sitting in the same folder first.
```

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

sns.set_theme(style="whitegrid")
CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)


def section(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def save_chart(fig, name):
    path = os.path.join(CHART_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved chart -> {path}")

In [3]:
section("1. DATA LOADING & CLEANING")

df = pd.read_csv("raw_customer_sales.csv")

print("\n-- First 10 records --")
print(df.head(10))

print("\n-- Last 10 records --")
print(df.tail(10))

print(f"\n-- Shape: {df.shape[0]} rows, {df.shape[1]} columns --")

print("\n-- Dataset info --")
df.info()

print("\n-- Column names --")
print(list(df.columns))

print("\n-- Summary statistics (numeric) --")
print(df.describe())

print("\n-- Missing values per column --")
missing = df.isnull().sum()
print(missing[missing > 0])

print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

# --- Handle missing values ---
for col in ["Customer Rating", "Delivery Days", "Discount (%)"]:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in ["City", "Payment Method"]:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)

print("\n-- Missing values after handling --")
print(df.isnull().sum().sum(), "remaining missing cells")

# --- Duplicates ---
dupe_count = df.duplicated().sum()
print(f"\nDuplicate transactions found: {dupe_count}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df)}")

# --- Invalid value detection ---
invalid_age = df[(df["Age"] < 10) | (df["Age"] > 100)]
print(f"\nInvalid ages found: {len(invalid_age)}")

invalid_price = df[(df["Unit Price"] < 0) | (df["Total Amount"] < 0)]
print(f"Negative price/total rows found: {len(invalid_price)}")

invalid_discount = df[(df["Discount (%)"] < 0) | (df["Discount (%)"] > 100)]
print(f"Invalid discount percentages found: {len(invalid_discount)}")

invalid_rating = df[(df["Customer Rating"] < 1) | (df["Customer Rating"] > 5)]
print(f"Invalid customer ratings found: {len(invalid_rating)}")

# --- Fix invalid values ---
age_median = df.loc[(df["Age"] >= 10) & (df["Age"] <= 100), "Age"].median()
df.loc[(df["Age"] < 10) | (df["Age"] > 100), "Age"] = age_median

df["Unit Price"] = df["Unit Price"].abs()
df["Total Amount"] = df["Total Amount"].abs()

df.loc[df["Discount (%)"] < 0, "Discount (%)"] = 0
df.loc[df["Discount (%)"] > 100, "Discount (%)"] = 100

rating_median = df.loc[(df["Customer Rating"] >= 1) & (df["Customer Rating"] <= 5), "Customer Rating"].median()
df.loc[(df["Customer Rating"] < 1) | (df["Customer Rating"] > 5), "Customer Rating"] = rating_median

# --- Type conversions ---
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"])
num_cols = ["Age", "Quantity", "Unit Price", "Discount (%)", "Total Amount",
            "Customer Rating", "Delivery Days"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["Age"] = df["Age"].astype(int)
df["Delivery Days"] = df["Delivery Days"].astype(int)

print("\n-- Cleaned dataset verification --")
df.info()
print(f"\nFinal cleaned shape: {df.shape}")


1. DATA LOADING & CLEANING

-- First 10 records --
  Transaction ID Customer ID   Customer Name  Age  Gender        City       Product Name            Category  Quantity  Unit Price  \
0       TXN00038    CUST0028      Sara Ahmed   39    Male  Gujranwala       Planner 2026  Books & Stationery         3    14880.91   
1       TXN00171    CUST0032       Asad Khan   26    Male  Rawalpindi  Bluetooth Speaker         Electronics         2    17884.37   
2       TXN00257    CUST0071      Nadia Khan   41    Male     Karachi          Honey Jar           Groceries         7    17142.83   
3       TXN00061    CUST0052      Warda Khan   26    Male   Islamabad   Wireless Earbuds         Electronics         6    13149.68   
4       TXN00154    CUST0076    Iqra Hussain   52  Female     Karachi         Hair Serum              Beauty         7     8812.63   
5       TXN00182    CUST0031   Fatima Farooq   22    Male    Peshawar     Green Tea Pack           Groceries         1     5792.49   
6       TX

In [4]:
section("2. CUSTOMER ANALYSIS")

n_unique_customers = df["Customer ID"].nunique()
print(f"Total unique customers: {n_unique_customers}")

oldest = df.loc[df["Age"].idxmax()]
youngest = df.loc[df["Age"].idxmin()]
print(f"Oldest customer: {oldest['Customer Name']} (Age {oldest['Age']})")
print(f"Youngest customer: {youngest['Customer Name']} (Age {youngest['Age']})")

print(f"Average customer age: {df['Age'].mean():.1f}")

spend_by_customer = df.groupby(["Customer ID", "Customer Name"])["Total Amount"].sum().sort_values(ascending=False)
print(f"\nAverage spending per customer: {spend_by_customer.mean():,.2f}")
print(f"Highest-spending customer: {spend_by_customer.index[0]} -> {spend_by_customer.iloc[0]:,.2f}")
print(f"Lowest-spending customer: {spend_by_customer.index[-1]} -> {spend_by_customer.iloc[-1]:,.2f}")

print("\nTop 10 highest-spending customers:")
print(spend_by_customer.head(10))

purchase_counts = df.groupby("Customer ID").size()
frequent_customers = purchase_counts[purchase_counts > 5]
print(f"\nCustomers with more than 5 purchases: {len(frequent_customers)}")

avg_spend = spend_by_customer.mean()
above_avg_customers = spend_by_customer[spend_by_customer > avg_spend]
print(f"Customers with spending above average ({avg_spend:,.2f}): {len(above_avg_customers)}")

print(f"\nAverage customer rating: {df['Customer Rating'].mean():.2f}")
high_rating_customers = df[df["Customer Rating"] > 4.5]["Customer ID"].nunique()
print(f"Customers with ratings above 4.5: {high_rating_customers}")

gender_counts = df.groupby("Customer ID")["Gender"].first().value_counts()
print("\nCustomers by gender:")
print(gender_counts)

avg_spend_gender = df.groupby("Gender")["Total Amount"].mean().sort_values(ascending=False)
print("\nAverage spending by gender:")
print(avg_spend_gender)
print(f"Gender with highest average spending: {avg_spend_gender.index[0]}")


2. CUSTOMER ANALYSIS
Total unique customers: 86
Oldest customer: Maria Chaudhry (Age 64)
Youngest customer: Hamza Farooq (Age 18)
Average customer age: 41.3

Average spending per customer: 166,176.88
Highest-spending customer: ('CUST0066', 'Sana Malik') -> 505,239.87
Lowest-spending customer: ('CUST0064', 'Mehak Raza') -> 6,342.27

Top 10 highest-spending customers:
Customer ID  Customer Name 
CUST0066     Sana Malik        505239.87
CUST0038     Junaid Raza       489497.54
CUST0069     Maria Khan        481282.53
CUST0045     Mehak Raza        423141.24
CUST0081     Amina Baig        396197.35
CUST0060     Hina Iqbal        379475.18
CUST0031     Fatima Farooq     362917.89
CUST0034     Zeeshan Farooq    330204.37
CUST0055     Maria Ahmed       323933.84
CUST0004     Asad Javed        321251.28
Name: Total Amount, dtype: float64

Customers with more than 5 purchases: 12
Customers with spending above average (166,176.88): 35

Average customer rating: 2.98
Customers with ratings above 

In [5]:
section("3. PRODUCT & CATEGORY ANALYSIS")

print(f"Number of distinct products sold: {df['Product Name'].nunique()}")

product_freq = df["Product Name"].value_counts()
print(f"\nMost frequently purchased product: {product_freq.index[0]} ({product_freq.iloc[0]} orders)")
print(f"Least frequently purchased product: {product_freq.index[-1]} ({product_freq.iloc[-1]} orders)")

qty_by_product = df.groupby("Product Name")["Quantity"].sum().sort_values(ascending=False)
sales_by_product = df.groupby("Product Name")["Total Amount"].sum().sort_values(ascending=False)

print("\nTop 10 products by sales:")
print(sales_by_product.head(10))

print("\nTop 10 products by quantity sold:")
print(qty_by_product.head(10))

category_counts = df["Category"].value_counts()
print("\nProducts per category (order count):")
print(category_counts)

avg_sales_category = df.groupby("Category")["Total Amount"].mean().sort_values(ascending=False)
print("\nAverage sales by category:")
print(avg_sales_category)

total_rev_category = df.groupby("Category")["Total Amount"].sum().sort_values(ascending=False)
print("\nTotal revenue by category:")
print(total_rev_category)

print(f"\nMost popular category (by orders): {category_counts.index[0]}")
print(f"Category generating highest revenue: {total_rev_category.index[0]}")

avg_txn_value_category = df.groupby("Category")["Total Amount"].mean().sort_values(ascending=False)
print(f"Category with highest average transaction value: {avg_txn_value_category.index[0]}")

avg_rating_category = df.groupby("Category")["Customer Rating"].mean().sort_values(ascending=False)
print("\nAverage customer rating by category:")
print(avg_rating_category)


3. PRODUCT & CATEGORY ANALYSIS
Number of distinct products sold: 40

Most frequently purchased product: Denim Jeans (16 orders)
Least frequently purchased product: Winter Jacket (2 orders)

Top 10 products by sales:
Product Name
Denim Jeans        914550.85
Face Wash          643793.89
Power Bank         570723.26
Cricket Bat        561759.51
Table Lamp         522997.91
Desk Organizer     505782.41
Green Tea Pack     474947.13
Lipstick Set       461798.88
Sketch Pens Set    449836.71
Planner 2026       440314.68
Name: Total Amount, dtype: float64

Top 10 products by quantity sold:
Product Name
Denim Jeans        73
Face Wash          50
Green Tea Pack     47
Cricket Bat        45
Sketch Pens Set    42
Desk Organizer     41
Table Lamp         41
Power Bank         41
Lipstick Set       40
Moisturizer        39
Name: Quantity, dtype: int64

Products per category (order count):
Category
Beauty                46
Clothing              46
Books & Stationery    45
Groceries             44
H

In [6]:
section("4. SALES & REVENUE ANALYSIS")

df["Gross Sales"] = df["Quantity"] * df["Unit Price"]
df["Discount Amount"] = df["Gross Sales"] * df["Discount (%)"] / 100
df["Net Sales"] = df["Gross Sales"] - df["Discount Amount"]

print(f"Total gross sales: {df['Gross Sales'].sum():,.2f}")
print(f"Total discount amount: {df['Discount Amount'].sum():,.2f}")
print(f"Total net sales: {df['Net Sales'].sum():,.2f}")
print(f"Average transaction value (net sales): {df['Net Sales'].mean():,.2f}")

max_net = df.loc[df["Net Sales"].idxmax()]
min_net = df.loc[df["Net Sales"].idxmin()]
print(f"\nHighest net-sales transaction: {max_net['Transaction ID']} -> {max_net['Net Sales']:,.2f}")
print(f"Lowest net-sales transaction: {min_net['Transaction ID']} -> {min_net['Net Sales']:,.2f}")

print("\nTop 10 transactions by net sales:")
print(df.nlargest(10, "Net Sales")[["Transaction ID", "Customer Name", "Net Sales"]])

high_discount_txns = df[df["Discount (%)"] > 20]
print(f"\nTransactions with discount > 20%: {len(high_discount_txns)}")

high_discount_amt_txns = df[df["Discount Amount"] > 5000]
print(f"Transactions with discount amount > 5,000: {len(high_discount_amt_txns)}")

above_avg_net = df[df["Net Sales"] > df["Net Sales"].mean()]
print(f"Transactions with net sales above average: {len(above_avg_net)}")

total_sales_category = df.groupby("Category")["Net Sales"].sum().sort_values(ascending=False)
print("\nTotal net sales by category:")
print(total_sales_category)

avg_txn_value_by_cat = df.groupby("Category")["Net Sales"].mean().sort_values(ascending=False)
print("\nAverage transaction value by category:")
print(avg_txn_value_by_cat)

print(f"\nCategory with highest net sales: {total_sales_category.index[0]}")


4. SALES & REVENUE ANALYSIS
Total gross sales: 15,996,769.85
Total discount amount: 1,791,207.95
Total net sales: 14,205,561.90
Average transaction value (net sales): 47,194.56

Highest net-sales transaction: TXN00095 -> 171,683.96
Lowest net-sales transaction: TXN00199 -> 0.00

Top 10 transactions by net sales:
    Transaction ID   Customer Name    Net Sales
22        TXN00095     Ali Qureshi  171683.9600
101       TXN00189  Hamza Chaudhry  163404.2200
157       TXN00233    Ayesha Ahmed  157625.4820
141       TXN00097   Fatima Farooq  157410.0360
132       TXN00231    Asad Hussain  156798.6000
85        TXN00105      Mehak Raza  153017.3400
32        TXN00254    Salman Ahmed  148101.0000
276       TXN00190     Junaid Raza  142889.2920
122       TXN00090     Junaid Raza  142266.1765
143       TXN00287  Warda Chaudhry  141528.3780

Transactions with discount > 20%: 33
Transactions with discount amount > 5,000: 112
Transactions with net sales above average: 116

Total net sales by categ

In [7]:
section("5. CITY & LOCATION ANALYSIS")

txns_by_city = df["City"].value_counts()
print("Transactions by city:")
print(txns_by_city)
print(f"\nCity with highest number of transactions: {txns_by_city.index[0]}")
print(f"City with lowest number of transactions: {txns_by_city.index[-1]}")

total_sales_city = df.groupby("City")["Net Sales"].sum().sort_values(ascending=False)
avg_sales_city = df.groupby("City")["Net Sales"].mean().sort_values(ascending=False)
print("\nTotal sales by city:")
print(total_sales_city)
print("\nAverage sales by city:")
print(avg_sales_city)

print(f"\nCity generating highest revenue: {total_sales_city.index[0]}")
print(f"City with highest average transaction value: {avg_sales_city.index[0]}")

avg_rating_city = df.groupby("City")["Customer Rating"].mean().sort_values(ascending=False)
print("\nAverage customer rating by city:")
print(avg_rating_city)
print(f"City with highest customer satisfaction: {avg_rating_city.index[0]}")

print("\nTop 10 cities by revenue:")
print(total_sales_city.head(10))


5. CITY & LOCATION ANALYSIS
Transactions by city:
City
Multan        46
Rawalpindi    38
Gujranwala    37
Sialkot       35
Karachi       34
Peshawar      31
Lahore        26
Quetta        25
Faisalabad    15
Islamabad     14
Name: count, dtype: int64

City with highest number of transactions: Multan
City with lowest number of transactions: Islamabad

Total sales by city:
City
Multan        2.213878e+06
Gujranwala    2.009106e+06
Sialkot       1.743066e+06
Karachi       1.572378e+06
Rawalpindi    1.566165e+06
Peshawar      1.541879e+06
Quetta        1.429881e+06
Lahore        1.013245e+06
Faisalabad    6.278648e+05
Islamabad     4.880989e+05
Name: Net Sales, dtype: float64

Average sales by city:
City
Quetta        57195.246580
Gujranwala    54300.159162
Sialkot       49801.898629
Peshawar      49738.026290
Multan        48127.772652
Karachi       46246.423206
Faisalabad    41857.653833
Rawalpindi    41214.858184
Lahore        38970.974385
Islamabad     34864.206679
Name: Net Sales, dt

In [8]:
section("6. MEMBERSHIP ANALYSIS")

membership_counts = df.groupby("Customer ID")["Membership Type"].first().value_counts()
print("Customers by membership type:")
print(membership_counts)

avg_spend_membership = df.groupby("Membership Type")["Net Sales"].mean().sort_values(ascending=False)
print("\nAverage spending by membership type:")
print(avg_spend_membership)

avg_rating_membership = df.groupby("Membership Type")["Customer Rating"].mean().sort_values(ascending=False)
print("\nAverage rating by membership type:")
print(avg_rating_membership)

total_rev_membership = df.groupby("Membership Type")["Net Sales"].sum().sort_values(ascending=False)
print("\nTotal revenue by membership type:")
print(total_rev_membership)

print(f"\nMembership type generating highest revenue: {total_rev_membership.index[0]}")
print(f"Membership type with highest average spending: {avg_spend_membership.index[0]}")

purchase_freq_membership = df.groupby("Membership Type").size().sort_values(ascending=False)
print("\nPurchase frequency by membership type:")
print(purchase_freq_membership)

print(f"Membership type with highest customer rating: {avg_rating_membership.index[0]}")


6. MEMBERSHIP ANALYSIS
Customers by membership type:
Membership Type
Basic       34
Silver      28
Gold        15
Platinum     9
Name: count, dtype: int64

Average spending by membership type:
Membership Type
Basic       49433.443657
Silver      47916.108102
Gold        44072.322141
Platinum    40146.971350
Name: Net Sales, dtype: float64

Average rating by membership type:
Membership Type
Gold        3.219565
Basic       3.010236
Silver      2.865306
Platinum    2.830000
Name: Customer Rating, dtype: float64

Total revenue by membership type:
Membership Type
Basic       6.278047e+06
Silver      4.695779e+06
Gold        2.027327e+06
Platinum    1.204409e+06
Name: Net Sales, dtype: float64

Membership type generating highest revenue: Basic
Membership type with highest average spending: Basic

Purchase frequency by membership type:
Membership Type
Basic       127
Silver       98
Gold         46
Platinum     30
dtype: int64
Membership type with highest customer rating: Gold


In [9]:
section("7. PAYMENT METHOD ANALYSIS")

payment_counts = df["Payment Method"].value_counts()
print("Transactions by payment method:")
print(payment_counts)

total_sales_payment = df.groupby("Payment Method")["Net Sales"].sum().sort_values(ascending=False)
avg_txn_payment = df.groupby("Payment Method")["Net Sales"].mean().sort_values(ascending=False)

print("\nTotal sales by payment method:")
print(total_sales_payment)
print("\nAverage transaction value by payment method:")
print(avg_txn_payment)

print(f"\nMost commonly used payment method: {payment_counts.index[0]}")
print(f"Payment method generating highest revenue: {total_sales_payment.index[0]}")
print(f"Payment method with highest average transaction value: {avg_txn_payment.index[0]}")


7. PAYMENT METHOD ANALYSIS
Transactions by payment method:
Payment Method
Credit Card      94
Mobile Wallet    72
Debit Card       61
Cash             42
Bank Transfer    32
Name: count, dtype: int64

Total sales by payment method:
Payment Method
Credit Card      5.191455e+06
Debit Card       3.103578e+06
Mobile Wallet    2.809579e+06
Bank Transfer    1.560526e+06
Cash             1.540424e+06
Name: Net Sales, dtype: float64

Average transaction value by payment method:
Payment Method
Credit Card      55228.242293
Debit Card       50878.329434
Bank Transfer    48766.440125
Mobile Wallet    39021.929854
Cash             36676.761738
Name: Net Sales, dtype: float64

Most commonly used payment method: Credit Card
Payment method generating highest revenue: Credit Card
Payment method with highest average transaction value: Credit Card


In [10]:
section("8. ORDER & DELIVERY ANALYSIS")

status_counts = df["Order Status"].value_counts()
print("Orders by status:")
print(status_counts)
print(f"\nMost common order status: {status_counts.index[0]}")

print(f"Average delivery time: {df['Delivery Days'].mean():.2f} days")
print(f"Fastest delivery: {df['Delivery Days'].min()} day(s)")
print(f"Slowest delivery: {df['Delivery Days'].max()} day(s)")

fast_orders = df[df["Delivery Days"] < 3]
slow_orders = df[df["Delivery Days"] > 7]
print(f"\nOrders delivered in < 3 days: {len(fast_orders)}")
print(f"Orders taking > 7 days: {len(slow_orders)}")

avg_delivery_city = df.groupby("City")["Delivery Days"].mean().sort_values()
print("\nAverage delivery days by city:")
print(avg_delivery_city)

avg_delivery_category = df.groupby("Category")["Delivery Days"].mean().sort_values()
print("\nAverage delivery days by category:")
print(avg_delivery_category)
print(f"\nCategory with longest average delivery time: {avg_delivery_category.index[-1]}")

delivery_rating_corr = df["Delivery Days"].corr(df["Customer Rating"])
print(f"\nCorrelation between delivery time and customer rating: {delivery_rating_corr:.3f}")


8. ORDER & DELIVERY ANALYSIS
Orders by status:
Order Status
Delivered     248
Cancelled      25
Returned       17
Processing     11
Name: count, dtype: int64

Most common order status: Delivered
Average delivery time: 5.82 days
Fastest delivery: 1 day(s)
Slowest delivery: 11 day(s)

Orders delivered in < 3 days: 62
Orders taking > 7 days: 103

Average delivery days by city:
City
Quetta        4.760000
Lahore        4.961538
Islamabad     5.500000
Karachi       5.588235
Gujranwala    5.621622
Sialkot       6.114286
Peshawar      6.193548
Multan        6.195652
Rawalpindi    6.210526
Faisalabad    6.866667
Name: Delivery Days, dtype: float64

Average delivery days by category:
Category
Electronics           5.189189
Sports & Fitness      5.550000
Books & Stationery    5.711111
Groceries             5.772727
Beauty                6.021739
Home & Kitchen        6.186047
Clothing              6.195652
Name: Delivery Days, dtype: float64

Category with longest average delivery time: Clothin

In [11]:
section("9. DATE-BASED SALES ANALYSIS")

df["Year"] = df["Purchase Date"].dt.year
df["Month"] = df["Purchase Date"].dt.month
df["Month Name"] = df["Purchase Date"].dt.month_name()
df["Day of Week"] = df["Purchase Date"].dt.day_name()

txns_by_month = df["Month Name"].value_counts()
print("Transactions by month:")
print(txns_by_month)

sales_by_month = df.groupby("Month Name")["Net Sales"].sum()
month_order = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]
sales_by_month = sales_by_month.reindex(month_order).dropna()
avg_sales_by_month = df.groupby("Month Name")["Net Sales"].mean().reindex(month_order).dropna()

print("\nTotal sales by month:")
print(sales_by_month)
print("\nAverage sales by month:")
print(avg_sales_by_month)

print(f"\nMonth with highest sales: {sales_by_month.idxmax()}")
print(f"Month with lowest sales: {sales_by_month.idxmin()}")

sales_by_year = df.groupby("Year")["Net Sales"].sum()
print("\nTotal sales by year:")
print(sales_by_year)
print(f"Year with highest revenue: {sales_by_year.idxmax()}")

df["Is Weekend"] = df["Day of Week"].isin(["Saturday", "Sunday"])
weekend_vs_weekday = df.groupby("Is Weekend")["Net Sales"].sum()
print("\nWeekday vs weekend sales (False=weekday, True=weekend):")
print(weekend_vs_weekday)

dow_counts = df["Day of Week"].value_counts()
print("\nPurchases by day of week:")
print(dow_counts)
print(f"Day of week with highest number of purchases: {dow_counts.index[0]}")


9. DATE-BASED SALES ANALYSIS
Transactions by month:
Month Name
August       35
March        31
December     30
May          26
January      25
April        25
July         24
February     22
June         21
October      21
November     21
September    20
Name: count, dtype: int64

Total sales by month:
Month Name
January      9.931639e+05
February     1.177243e+06
March        1.556514e+06
April        1.187744e+06
May          1.514236e+06
June         1.125248e+06
July         1.227923e+06
August       1.600579e+06
September    7.443282e+05
October      6.306207e+05
November     8.730457e+05
December     1.574916e+06
Name: Net Sales, dtype: float64

Average sales by month:
Month Name
January      39726.554200
February     53511.057000
March        50210.126032
April        47509.772560
May          58239.838346
June         53583.251833
July         51163.467458
August       45730.830971
September    37216.411575
October      30029.554881
November     41573.602619
December     52497

In [12]:
section("10. ADVANCED CUSTOMER FILTERING")

big_spenders = spend_by_customer[spend_by_customer > 100_000]
print(f"Customers spending more than 100,000: {len(big_spenders)}")

freq_buyers = purchase_counts[purchase_counts > 5]
print(f"Customers with more than 5 transactions: {len(freq_buyers)}")

top_rated_customers = df[df["Customer Rating"] > 4.5]["Customer ID"].nunique()
print(f"Customers with ratings above 4.5: {top_rated_customers}")

young_high_spend = df[(df["Age"] < 25) & (df["Net Sales"] > df["Net Sales"].mean())]
print(f"Customers younger than 25 with spending above average: {len(young_high_spend)}")

platinum_big = df[(df["Membership Type"] == "Platinum") & (df["Net Sales"] > 50_000)]
print(f"Platinum members with transactions > 50,000: {len(platinum_big)}")

bulk_buyers = df[df["Quantity"] > 10]
print(f"Transactions with more than 10 items purchased: {len(bulk_buyers)}")

high_discount = df[df["Discount (%)"] > 25]
print(f"Transactions with discounts above 25%: {len(high_discount)}")

slow_delivery = df[df["Delivery Days"] > 7]
print(f"Orders with delivery time > 7 days: {len(slow_delivery)}")

avg_qty = qty_by_product.mean()
above_avg_products = qty_by_product[qty_by_product > avg_qty]
print(f"Products with quantity sold above average: {len(above_avg_products)}")

avg_cat_revenue = total_rev_category.mean()
above_avg_categories = total_rev_category[total_rev_category > avg_cat_revenue]
print(f"Categories with revenue above average category revenue: {list(above_avg_categories.index)}")


10. ADVANCED CUSTOMER FILTERING
Customers spending more than 100,000: 55
Customers with more than 5 transactions: 12
Customers with ratings above 4.5: 26
Customers younger than 25 with spending above average: 11
Platinum members with transactions > 50,000: 11
Transactions with more than 10 items purchased: 0
Transactions with discounts above 25%: 14
Orders with delivery time > 7 days: 103
Products with quantity sold above average: 20
Categories with revenue above average category revenue: ['Beauty', 'Clothing', 'Books & Stationery']


In [13]:
section("11. CUSTOMER SEGMENTATION")

cust_totals = df.groupby("Customer ID")["Net Sales"].sum()
q1, q2, q3 = cust_totals.quantile([0.25, 0.5, 0.75])


def segment(spend):
    if spend <= q1:
        return "Low Spenders"
    elif spend <= q2:
        return "Medium Spenders"
    elif spend <= q3:
        return "High Spenders"
    else:
        return "Premium Customers"


cust_segment_map = cust_totals.apply(segment)
df["Customer Segment"] = df["Customer ID"].map(cust_segment_map)

segment_counts = df.groupby("Customer ID")["Customer Segment"].first().value_counts()
print("Customers per segment:")
print(segment_counts)

avg_spend_segment = df.groupby("Customer Segment")["Net Sales"].mean().sort_values(ascending=False)
print("\nAverage spending per segment:")
print(avg_spend_segment)

avg_rating_segment = df.groupby("Customer Segment")["Customer Rating"].mean().sort_values(ascending=False)
print("\nAverage rating per segment:")
print(avg_rating_segment)

total_rev_segment = df.groupby("Customer Segment")["Net Sales"].sum().sort_values(ascending=False)
print("\nTotal revenue per segment:")
print(total_rev_segment)
print(f"\nSegment generating highest revenue: {total_rev_segment.index[0]}")

purchase_behavior = df.groupby("Customer Segment").agg(
    avg_quantity=("Quantity", "mean"),
    avg_discount=("Discount (%)", "mean"),
    avg_delivery_days=("Delivery Days", "mean"),
)
print("\nPurchasing behavior comparison across segments:")
print(purchase_behavior)


11. CUSTOMER SEGMENTATION
Customers per segment:
Customer Segment
Low Spenders         22
Premium Customers    22
Medium Spenders      21
High Spenders        21
Name: count, dtype: int64

Average spending per segment:
Customer Segment
Premium Customers    61641.965038
High Spenders        46997.093753
Medium Spenders      34388.142500
Low Spenders         23312.495784
Name: Net Sales, dtype: float64

Average rating per segment:
Customer Segment
Premium Customers    3.057143
Medium Spenders      3.035938
High Spenders        2.988889
Low Spenders         2.591892
Name: Customer Rating, dtype: float64

Total revenue per segment:
Customer Segment
Premium Customers    7.335394e+06
High Spenders        3.806765e+06
Medium Spenders      2.200841e+06
Low Spenders         8.625623e+05
Name: Net Sales, dtype: float64

Segment generating highest revenue: Premium Customers

Purchasing behavior comparison across segments:
                   avg_quantity  avg_discount  avg_delivery_days
Customer 

In [14]:
section("DATA VISUALIZATION")

fig, ax = plt.subplots(figsize=(9, 5))
total_rev_category.sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Total Sales by Category"); ax.set_xlabel("Net Sales"); ax.set_ylabel("Category")
save_chart(fig, "01_sales_by_category.png")

fig, ax = plt.subplots(figsize=(9, 5))
total_sales_city.plot(kind="bar", ax=ax, color="#55A868")
ax.set_title("Revenue by City"); ax.set_xlabel("City"); ax.set_ylabel("Net Sales")
plt.xticks(rotation=45, ha="right")
save_chart(fig, "02_revenue_by_city.png")

fig, ax = plt.subplots(figsize=(10, 5))
sales_by_month.plot(kind="line", marker="o", ax=ax, color="#C44E52")
ax.set_title("Monthly Sales Trend"); ax.set_xlabel("Month"); ax.set_ylabel("Net Sales")
plt.xticks(rotation=45, ha="right")
save_chart(fig, "03_monthly_sales_trend.png")

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df["Age"], bins=15, kde=True, ax=ax, color="#8172B2")
ax.set_title("Customer Age Distribution"); ax.set_xlabel("Age"); ax.set_ylabel("Frequency")
save_chart(fig, "04_customer_age_distribution.png")

fig, ax = plt.subplots(figsize=(6, 5))
df.groupby("Gender")["Net Sales"].sum().plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_title("Sales by Gender"); ax.set_xlabel("Gender"); ax.set_ylabel("Net Sales")
plt.xticks(rotation=0)
save_chart(fig, "05_sales_by_gender.png")

fig, ax = plt.subplots(figsize=(7, 5))
total_rev_membership.plot(kind="bar", ax=ax, color="#64B5CD")
ax.set_title("Revenue by Membership Type"); ax.set_xlabel("Membership Type"); ax.set_ylabel("Net Sales")
plt.xticks(rotation=0)
save_chart(fig, "06_revenue_by_membership.png")

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x="Payment Method", order=payment_counts.index, ax=ax, hue="Payment Method",
              palette="pastel", legend=False)
ax.set_title("Payment Method Distribution"); ax.set_xlabel("Payment Method"); ax.set_ylabel("Number of Transactions")
plt.xticks(rotation=30, ha="right")
save_chart(fig, "07_payment_method_distribution.png")

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df["Customer Rating"], bins=10, kde=True, ax=ax, color="#937860")
ax.set_title("Customer Rating Distribution"); ax.set_xlabel("Customer Rating"); ax.set_ylabel("Frequency")
save_chart(fig, "08_customer_rating_distribution.png")

fig, ax = plt.subplots(figsize=(9, 6))
sales_by_product.head(10).sort_values().plot(kind="barh", ax=ax, color="#DA8BC3")
ax.set_title("Top 10 Products by Revenue"); ax.set_xlabel("Total Sales"); ax.set_ylabel("Product")
save_chart(fig, "09_top10_products_by_revenue.png")

fig, ax = plt.subplots(figsize=(7, 7))
segment_counts.plot(kind="pie", ax=ax, autopct="%1.1f%%", startangle=90,
                     colors=sns.color_palette("Set2"))
ax.set_title("Customer Segment Distribution"); ax.set_ylabel("")
save_chart(fig, "10_customer_segment_distribution.png")

fig, ax = plt.subplots(figsize=(8, 6))
corr_cols = ["Quantity", "Unit Price", "Discount (%)", "Total Amount", "Age",
             "Customer Rating", "Delivery Days", "Net Sales"]
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Heatmap")
save_chart(fig, "11_correlation_heatmap.png")


DATA VISUALIZATION
  saved chart -> charts/01_sales_by_category.png
  saved chart -> charts/02_revenue_by_city.png
  saved chart -> charts/03_monthly_sales_trend.png
  saved chart -> charts/04_customer_age_distribution.png
  saved chart -> charts/05_sales_by_gender.png
  saved chart -> charts/06_revenue_by_membership.png
  saved chart -> charts/07_payment_method_distribution.png
  saved chart -> charts/08_customer_rating_distribution.png
  saved chart -> charts/09_top10_products_by_revenue.png
  saved chart -> charts/10_customer_segment_distribution.png
  saved chart -> charts/11_correlation_heatmap.png


In [15]:
section("ADVANCED STATISTICAL ANALYSIS")

print(f"Mean transaction value: {df['Net Sales'].mean():,.2f}")
print(f"Median transaction value: {df['Net Sales'].median():,.2f}")
print(f"Minimum sales: {df['Net Sales'].min():,.2f}")
print(f"Maximum sales: {df['Net Sales'].max():,.2f}")
print(f"Standard deviation of sales: {df['Net Sales'].std():,.2f}")
print(f"Variance of sales: {df['Net Sales'].var():,.2f}")

corr_qty_sales = df["Quantity"].corr(df["Net Sales"])
corr_discount_net = df["Discount (%)"].corr(df["Net Sales"])
corr_age_spend = df["Age"].corr(df["Net Sales"])
corr_rating_spend = df["Customer Rating"].corr(df["Net Sales"])

print(f"\nCorrelation (Quantity vs Total Sales): {corr_qty_sales:.3f}")
print(f"Correlation (Discount vs Net Sales): {corr_discount_net:.3f}")
print(f"Correlation (Age vs Spending): {corr_age_spend:.3f}")
print(f"Correlation (Rating vs Spending): {corr_rating_spend:.3f}")

corr_pairs = corr_matrix.where(~np.eye(len(corr_matrix), dtype=bool)).stack()
strongest_pos = corr_pairs.idxmax()
strongest_neg = corr_pairs.idxmin()
print(f"\nStrongest positive correlation: {strongest_pos} = {corr_pairs.max():.3f}")
print(f"Strongest negative correlation: {strongest_neg} = {corr_pairs.min():.3f}")


ADVANCED STATISTICAL ANALYSIS
Mean transaction value: 47,194.56
Median transaction value: 37,581.07
Minimum sales: 0.00
Maximum sales: 171,683.96
Standard deviation of sales: 39,532.82
Variance of sales: 1,562,843,736.16

Correlation (Quantity vs Total Sales): 0.572
Correlation (Discount vs Net Sales): -0.120
Correlation (Age vs Spending): 0.081
Correlation (Rating vs Spending): 0.088

Strongest positive correlation: ('Total Amount', 'Net Sales') = 0.982
Strongest negative correlation: ('Age', 'Delivery Days') = -0.140


In [16]:
section("SAVING OUTPUTS")

df.to_csv("analyzed_customer_sales.csv", index=False)
print("Saved analyzed_customer_sales.csv")

best_product = sales_by_product.index[0]
best_category = total_rev_category.index[0]
best_city = total_sales_city.index[0]
best_payment = payment_counts.index[0]
best_membership = total_rev_membership.index[0]
best_segment = total_rev_segment.index[0]

summary_lines = [
    "CUSTOMER SHOPPING & SALES ANALYTICS - SUMMARY REPORT",
    "=" * 55,
    f"Total Customers: {n_unique_customers}",
    f"Total Transactions: {len(df)}",
    f"Total Gross Sales: {df['Gross Sales'].sum():,.2f}",
    f"Total Discounts: {df['Discount Amount'].sum():,.2f}",
    f"Total Net Sales: {df['Net Sales'].sum():,.2f}",
    f"Average Transaction Value: {df['Net Sales'].mean():,.2f}",
    f"Best-Selling Product: {best_product}",
    f"Best-Performing Category: {best_category}",
    f"Highest-Revenue City: {best_city}",
    f"Most Common Payment Method: {best_payment}",
    f"Highest-Revenue Membership Type: {best_membership}",
    f"Best Customer Segment: {best_segment}",
]
report_text = "\n".join(summary_lines)
with open("summary_report.txt", "w") as f:
    f.write(report_text)

print("\n" + report_text)
print("\nSaved summary_report.txt")
print(f"All charts saved in ./{CHART_DIR}/")
print("\nPipeline complete.")


SAVING OUTPUTS
Saved analyzed_customer_sales.csv

CUSTOMER SHOPPING & SALES ANALYTICS - SUMMARY REPORT
Total Customers: 86
Total Transactions: 301
Total Gross Sales: 15,996,769.85
Total Discounts: 1,791,207.95
Total Net Sales: 14,205,561.90
Average Transaction Value: 47,194.56
Best-Selling Product: Denim Jeans
Best-Performing Category: Beauty
Highest-Revenue City: Multan
Most Common Payment Method: Credit Card
Highest-Revenue Membership Type: Basic
Best Customer Segment: Premium Customers

Saved summary_report.txt
All charts saved in ./charts/

Pipeline complete.


In [17]:
import os
import zipfile

with zipfile.ZipFile("customer_analytics_outputs.zip", "w") as zf:
    zf.write("analyzed_customer_sales.csv")
    zf.write("summary_report.txt")
    for chart_file in os.listdir(CHART_DIR):
        zf.write(os.path.join(CHART_DIR, chart_file))

print("Bundled: customer_analytics_outputs.zip")

try:
    from google.colab import files
    files.download("customer_analytics_outputs.zip")
except ImportError:
    print("Not running in Colab — find 'customer_analytics_outputs.zip' "
          "in your working directory and download it from the Jupyter file browser.")

Bundled: customer_analytics_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>